# End-to-End AML Detection Pipeline

This notebook demonstrates a complete AML detection pipeline using the
`aml-analytics` toolkit — from raw transactions to a unified risk report.

The pipeline combines all four detection methods:
1. **Velocity analysis** — transaction frequency and burst activity
2. **SAR pattern matching** — FATF/FinCEN typology rules
3. **Graph analysis** — network centrality and structuring rings
4. **Anomaly scoring** — unsupervised ensemble model

Each account receives a combined risk tier: High, Medium, or Low.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from aml.synthetic import generate_transactions
from aml.velocity import compute_velocity
from aml.patterns import run_all
from aml.graph import build_network, detect_structuring, centrality_score
from aml.anomaly import ensemble_score

print("All modules loaded")

## 1. Generate transaction data

In [ ]:
df = generate_transactions(n=1000, seed=42)
accounts = pd.DataFrame({'account_id': df['sender_id'].unique()})
print(f"Transactions : {len(df):,}")
print(f"Accounts     : {len(accounts):,}")

## 2. Run each detection module

In [ ]:
# Velocity
velocity = compute_velocity(df, window_hours=24, min_transactions=3)
velocity_accounts = set(velocity['account_id'].tolist())
print(f"Velocity alerts     : {len(velocity_accounts)}")

# SAR patterns
patterns = run_all(df)
structuring_accounts = set(patterns['structuring']['sender_id'].tolist()) if len(patterns['structuring']) > 0 else set()
smurfing_accounts    = set(patterns['smurfing']['receiver_id'].tolist()) if len(patterns['smurfing']) > 0 else set()
cash_accounts        = set(patterns['cash_intensive']['account_id'].tolist()) if len(patterns['cash_intensive']) > 0 else set()
print(f"Structuring alerts  : {len(structuring_accounts)}")
print(f"Smurfing alerts     : {len(smurfing_accounts)}")
print(f"Cash-intensive      : {len(cash_accounts)}")

# Graph centrality
G = build_network(df)
centrality = centrality_score(G)
high_centrality = set(centrality[centrality['risk_score'] > centrality['risk_score'].quantile(0.80)]['node_id'].tolist())
print(f"High centrality     : {len(high_centrality)}")

# Anomaly scoring
anomaly = ensemble_score(df)
anomaly_accounts = set(anomaly[anomaly['is_anomaly']]['account_id'].tolist())
print(f"Anomaly alerts      : {len(anomaly_accounts)}")

## 3. Build unified risk report
Each account gets a risk score based on how many detection methods flagged it.

In [ ]:
def assign_risk_tier(score):
    if score >= 3:
        return 'High'
    elif score >= 1:
        return 'Medium'
    return 'Low'

risk_rows = []
for _, row in accounts.iterrows():
    acc = row['account_id']
    flags = {
        'velocity':      acc in velocity_accounts,
        'structuring':   acc in structuring_accounts,
        'smurfing':      acc in smurfing_accounts,
        'cash_intensive': acc in cash_accounts,
        'high_centrality': acc in high_centrality,
        'anomaly':       acc in anomaly_accounts,
    }
    flag_count = sum(flags.values())
    risk_rows.append({
        'account_id':   acc,
        'flag_count':   flag_count,
        'risk_tier':    assign_risk_tier(flag_count),
        **flags
    })

risk_report = pd.DataFrame(risk_rows).sort_values('flag_count', ascending=False).reset_index(drop=True)
print(f"Risk tier breakdown:")
print(risk_report['risk_tier'].value_counts().to_string())
print(f"\nTop 10 highest risk accounts:")
print(risk_report.head(10)[['account_id','flag_count','risk_tier']].to_string(index=False))

## 4. Visualize the risk report

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Risk tier pie chart
tier_counts = risk_report['risk_tier'].value_counts()
colors_pie = {'High': 'crimson', 'Medium': 'darkorange', 'Low': 'steelblue'}
axes[0].pie(
    tier_counts,
    labels=tier_counts.index,
    colors=[colors_pie[t] for t in tier_counts.index],
    autopct='%1.1f%%',
    startangle=90
)
axes[0].set_title('Account Risk Tier Distribution', fontsize=12)

# Flag count distribution
flag_counts = risk_report['flag_count'].value_counts().sort_index()
axes[1].bar(flag_counts.index, flag_counts.values, color='steelblue', edgecolor='white')
axes[1].set_xlabel('Number of Detection Methods That Flagged Account', fontsize=10)
axes[1].set_ylabel('Number of Accounts', fontsize=10)
axes[1].set_title('Detection Method Overlap', fontsize=12)
axes[1].set_xticks(flag_counts.index)

plt.suptitle('End-to-End AML Pipeline — Risk Report Summary', fontsize=13)
plt.tight_layout()
plt.savefig('risk_report.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Heatmap of flags per account (top 20)
top20 = risk_report.head(20)
flag_cols = ['velocity','structuring','smurfing','cash_intensive','high_centrality','anomaly']
flag_matrix = top20[flag_cols].astype(int)

fig, ax = plt.subplots(figsize=(11, 7))
im = ax.imshow(flag_matrix.values, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)
ax.set_xticks(range(len(flag_cols)))
ax.set_xticklabels(flag_cols, rotation=30, ha='right', fontsize=10)
ax.set_yticks(range(len(top20)))
ax.set_yticklabels(top20['account_id'], fontsize=8)
ax.set_title('Top 20 High-Risk Accounts — Detection Method Heatmap\n(Green = Flagged)', fontsize=12)
plt.colorbar(im, ax=ax, ticks=[0,1], label='Flagged')
plt.tight_layout()
plt.savefig('risk_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# Heatmap of flags per account (top 20)
top20 = risk_report.head(20)
flag_cols = ['velocity','structuring','smurfing','cash_intensive','high_centrality','anomaly']
flag_matrix = top20[flag_cols].astype(int)

fig, ax = plt.subplots(figsize=(11, 7))
im = ax.imshow(flag_matrix.values, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)
ax.set_xticks(range(len(flag_cols)))
ax.set_xticklabels(flag_cols, rotation=30, ha='right', fontsize=10)
ax.set_yticks(range(len(top20)))
ax.set_yticklabels(top20['account_id'], fontsize=8)
ax.set_title('Top 20 High-Risk Accounts — Detection Method Heatmap\n(Green = Flagged)', fontsize=12)
plt.colorbar(im, ax=ax, ticks=[0,1], label='Flagged')
plt.tight_layout()
plt.savefig('risk_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
risk_report.to_csv('aml_risk_report.csv', index=False)
high_risk = risk_report[risk_report['risk_tier'] == 'High']
print(f"Risk report exported: {len(risk_report)} accounts")
print(f"High risk accounts  : {len(high_risk)}")
print(f"\nHigh risk account list:")
print(high_risk[['account_id','flag_count']].to_string(index=False))

## Summary

This pipeline demonstrates how combining multiple detection methods provides
a more complete view of account risk than any single method alone.

| Method | Strength |
|---|---|
| Velocity | Catches rapid fund movement and burst activity |
| SAR patterns | Explainable, regulatory-grounded alerts |
| Graph centrality | Identifies funnel accounts in laundering networks |
| Anomaly scoring | Catches novel patterns rules-based systems miss |

Accounts flagged by 3+ methods warrant priority SAR review.

Toolkit: [github.com/Bhavesh0205/aml-analytics](https://github.com/Bhavesh0205/aml-analytics)